# 01 Data Audit

This notebook starts the A/B testing case study by validating the raw dataset, checking basic data quality, and reviewing whether the randomized experiment groups look reasonably balanced before outcome analysis.

Source dataset: MineThatData E-Mail Analytics challenge.

## Goals

- Confirm the dataset loads correctly.
- Check row counts, columns, missing values, and category labels.
- Create a cleaned working file for later notebooks.
- Review randomization balance across pre-treatment customer attributes.
- Export starter summary tables for the project report.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONTROL_GROUP, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, RAW_DATA_PATH
from src.data_checks import load_raw_data, validate_raw_data
from src.metrics import group_summary

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT

WindowsPath('C:/.projects/ab-testing')

## Load Raw Data

In [2]:
df = load_raw_data(RAW_DATA_PATH)
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.4400,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0000
1,6,3) $200 - $350,329.0800,1,1,Rural,1,Web,No E-Mail,0,0,0.0000
2,7,2) $100 - $200,180.6500,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0000
3,9,5) $500 - $750,675.8300,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0000
4,2,1) $0 - $100,45.3400,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0000


In [3]:
df.shape

(64000, 12)

In [4]:
df.dtypes.to_frame("dtype")

,dtype
recency,int64
history_segment,str
history,float64
mens,int64
womens,int64
zip_code,str
newbie,int64
channel,str
segment,str
visit,int64


## Data Quality Checks

In [5]:
checks = validate_raw_data(df)
quality_summary = pd.DataFrame(
    [
        {"check": "rows", "value": checks["rows"]},
        {"check": "columns", "value": checks["columns"]},
        {"check": "missing_columns", "value": ", ".join(checks["missing_columns"]) or "None"},
        {"check": "extra_columns", "value": ", ".join(checks["extra_columns"]) or "None"},
        {"check": "experiment_groups", "value": ", ".join(checks["segments"])},
        {"check": "zip_code_values", "value": ", ".join(checks["zip_code_values"])},
        {"check": "total_missing_values", "value": sum(checks["missing_values"].values())},
    ]
)
quality_summary

,check,value
0,rows,64000
1,columns,12
2,missing_columns,None
3,extra_columns,None
4,experiment_groups,"Mens E-Mail, No E-Mail, Womens E-Mail"
5,zip_code_values,"Rural, Surburban, Urban"
6,total_missing_values,0


In [6]:
missing_by_column = df.isna().sum().rename("missing_values").to_frame()
missing_by_column["missing_rate"] = missing_by_column["missing_values"] / len(df)
missing_by_column

,missing_values,missing_rate
recency,0,0.0000
history_segment,0,0.0000
history,0,0.0000
mens,0,0.0000
womens,0,0.0000
zip_code,0,0.0000
newbie,0,0.0000
channel,0,0.0000
segment,0,0.0000
visit,0,0.0000


In [7]:
category_counts = {
    column: df[column].value_counts(dropna=False).to_frame("rows")
    for column in ["segment", "zip_code", "channel", "history_segment"]
}
category_counts["segment"]

,rows
segment,
Womens E-Mail,21387
Mens E-Mail,21307
No E-Mail,21306


In [8]:
category_counts["zip_code"]

,rows
zip_code,
Surburban,28776
Urban,25661
Rural,9563


The raw `zip_code` column contains `Surburban`. We will preserve the raw file and add a cleaned `zip_code_clean` field for analysis.

In [9]:
df_clean = df.copy()
df_clean["zip_code_clean"] = df_clean["zip_code"].replace({"Surburban": "Suburban"})
df_clean["is_treatment"] = (df_clean["segment"] != CONTROL_GROUP).astype(int)
df_clean["campaign_type"] = np.select(
    [
        df_clean["segment"].eq("Mens E-Mail"),
        df_clean["segment"].eq("Womens E-Mail"),
        df_clean["segment"].eq(CONTROL_GROUP),
    ],
    ["Mens", "Womens", "Control"],
    default="Unknown",
)

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
clean_path = INTERIM_DATA_DIR / "email_ab_test_clean.csv"
df_clean.to_csv(clean_path, index=False)
clean_path

WindowsPath('C:/.projects/ab-testing/data/interim/email_ab_test_clean.csv')

## Experiment Group Sizes

In [10]:
group_counts = (
    df_clean["segment"]
    .value_counts()
    .rename_axis("segment")
    .reset_index(name="customers")
)
group_counts["share"] = group_counts["customers"] / len(df_clean)
group_counts

,segment,customers,share
0,Womens E-Mail,21387,0.3342
1,Mens E-Mail,21307,0.3329
2,No E-Mail,21306,0.3329


## First Outcome Summary

This is a quick preview only. The formal hypothesis tests and revenue decision will happen in the next notebook.

In [11]:
summary = group_summary(df_clean).sort_values("segment")
summary

,segment,customers,visits,conversions,total_spend,spend_per_customer,visit_rate,conversion_rate,spend_per_converter
0,Mens E-Mail,21307,3894,267,30311.6900,1.4226,0.1828,0.0125,113.5269
1,No E-Mail,21306,2262,122,13908.3300,0.6528,0.1062,0.0057,114.0027
2,Womens E-Mail,21387,3238,189,23038.1100,1.0772,0.1514,0.0088,121.8948


In [12]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
quality_summary.to_csv(PROCESSED_DATA_DIR / "data_quality_summary.csv", index=False)
group_counts.to_csv(PROCESSED_DATA_DIR / "experiment_group_counts.csv", index=False)
summary.to_csv(PROCESSED_DATA_DIR / "group_outcome_summary.csv", index=False)

sorted(path.name for path in PROCESSED_DATA_DIR.glob("*.csv"))

['data_quality_summary.csv',
 'experiment_group_counts.csv',
 'group_outcome_summary.csv',
 'numeric_balance_smd.csv']

## Randomization Balance Check

Because this is a randomized experiment, the groups should look similar on pre-treatment variables. Small differences are normal. Large differences would be a warning sign.

In [13]:
pretreatment_numeric = ["recency", "history", "mens", "womens", "newbie"]
numeric_balance = df_clean.groupby("segment")[pretreatment_numeric].mean().T
numeric_balance

segment,Mens E-Mail,No E-Mail,Womens E-Mail
recency,5.7736,5.7497,5.7678
history,242.8359,240.8827,242.5366
mens,0.5509,0.5532,0.5489
womens,0.5514,0.5476,0.5501
newbie,0.5015,0.5020,0.5032


In [14]:
def standardized_mean_difference(data: pd.DataFrame, variable: str, treatment_group: str) -> float:
    control_values = data.loc[data["segment"].eq(CONTROL_GROUP), variable]
    treatment_values = data.loc[data["segment"].eq(treatment_group), variable]
    pooled_sd = np.sqrt((control_values.var(ddof=1) + treatment_values.var(ddof=1)) / 2)
    return (treatment_values.mean() - control_values.mean()) / pooled_sd

balance_rows = []
for treatment_group in ["Mens E-Mail", "Womens E-Mail"]:
    for variable in pretreatment_numeric:
        balance_rows.append(
            {
                "treatment_group": treatment_group,
                "variable": variable,
                "standardized_mean_difference": standardized_mean_difference(df_clean, variable, treatment_group),
            }
        )

balance_smd = pd.DataFrame(balance_rows)
balance_smd["abs_smd"] = balance_smd["standardized_mean_difference"].abs()
balance_smd.sort_values("abs_smd", ascending=False)

,treatment_group,variable,standardized_mean_difference,abs_smd
7,Womens E-Mail,mens,-0.0086,0.0086
1,Mens E-Mail,history,0.0076,0.0076
3,Mens E-Mail,womens,0.0076,0.0076
0,Mens E-Mail,recency,0.0068,0.0068
6,Womens E-Mail,history,0.0065,0.0065
5,Womens E-Mail,recency,0.0052,0.0052
8,Womens E-Mail,womens,0.0049,0.0049
2,Mens E-Mail,mens,-0.0046,0.0046
9,Womens E-Mail,newbie,0.0026,0.0026
4,Mens E-Mail,newbie,-0.0009,0.0009


In [15]:
balance_smd.to_csv(PROCESSED_DATA_DIR / "numeric_balance_smd.csv", index=False)
balance_smd["abs_smd"].max()

np.float64(0.008630731202832915)

Common rule of thumb: standardized mean differences below 0.10 are usually considered small for balance checks.

In [16]:
zip_balance = pd.crosstab(df_clean["zip_code_clean"], df_clean["segment"], normalize="columns")
zip_balance

segment,Mens E-Mail,No E-Mail,Womens E-Mail
zip_code_clean,,,
Rural,0.1522,0.1473,0.1487
Suburban,0.4459,0.4518,0.4512
Urban,0.4019,0.4009,0.4001


In [17]:
channel_balance = pd.crosstab(df_clean["channel"], df_clean["segment"], normalize="columns")
channel_balance

segment,Mens E-Mail,No E-Mail,Womens E-Mail
channel,,,
Multichannel,0.1209,0.1223,0.1206
Phone,0.4337,0.4378,0.4420
Web,0.4454,0.4399,0.4374


In [18]:
history_segment_balance = pd.crosstab(df_clean["history_segment"], df_clean["segment"], normalize="columns")
history_segment_balance

segment,Mens E-Mail,No E-Mail,Womens E-Mail
history_segment,,,
1) $0 - $100,0.3625,0.3573,0.3569
2) $100 - $200,0.2202,0.2270,0.2210
3) $200 - $350,0.1920,0.1898,0.1943
4) $350 - $500,0.0984,0.0997,0.1023
5) $500 - $750,0.0750,0.0775,0.0777
"6) $750 - $1,000",0.0302,0.0292,0.0277
"7) $1,000 +",0.0218,0.0195,0.0200


## Audit Notes

- The dataset loaded successfully with 64,000 rows and 12 raw columns.
- No missing values were found in the raw file.
- The experiment has one control group and two treatment groups.
- A cleaned working file was written to `data/interim/email_ab_test_clean.csv`.
- Group balance appears suitable for moving into formal experiment analysis, pending review of the standardized mean differences above.

Next notebook: `02_experiment_analysis.ipynb`.